In [1]:
# Imports (restricted to GPU 1)
import os
# Force notebook to see only GPU 1 (the second physical GPU). RESTART kernel after setting if torch was previously imported.
os.environ['CUDA_DEVICE_ORDER'] = 'PCI_BUS_ID'
os.environ['CUDA_VISIBLE_DEVICES'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print('libs imported')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'Visible GPUs: {torch.cuda.device_count()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'Visible GPU {i}: {torch.cuda.get_device_name(i)}')

/home/barradd/Documents/GitHub/machine_learning_chem_RGS/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


libs imported
CUDA available: True
Visible GPUs: 1
Visible GPU 0: Quadro GV100


In [2]:

# Load a small slice of the large training data
train_df = pd.read_csv('../data/hf_dataset_large_train.csv', nrows=1000)
val_df = pd.read_csv('../data/hf_dataset_large_val.csv', nrows=200)



In [3]:
# Load the pretrained tokenizer and model
model_name = 'ibm-research/materials.selfies-ted'
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Fix missing eos_token_id for BART sequence classification
if tokenizer.eos_token_id is None:
    tokenizer.eos_token_id = tokenizer.pad_token_id
    print(f'Set eos_token_id to {tokenizer.eos_token_id}')

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=1)
model.config.problem_type = 'regression'

# Ensure model config has eos_token_id set
if model.config.eos_token_id is None:
    model.config.eos_token_id = tokenizer.eos_token_id
    print(f'Set model config eos_token_id to {model.config.eos_token_id}')

Set eos_token_id to 3


Some weights of BartForSequenceClassification were not initialized from the model checkpoint at ibm-research/materials.selfies-ted and are newly initialized: ['classification_head.dense.bias', 'classification_head.dense.weight', 'classification_head.out_proj.bias', 'classification_head.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
def tokenize_fn(batch):
    texts = batch['text']
    max_len = 256  # pick a safe max; adjust as needed
    eos_id = tokenizer.eos_token_id
    # Use token 0 for padding (common padding token, different from EOS=3)
    pad_id = 0

    # Tokenize without adding special tokens; we'll manage EOS manually
    enc = tokenizer(texts, add_special_tokens=False, return_token_type_ids=False)
    input_ids = enc['input_ids']
    attention_mask = enc.get('attention_mask', None)

    out_ids = []
    out_mask = []
    for i, ids in enumerate(input_ids):
        # truncate so final position is reserved for EOS
        core = ids[:max_len - 1]
        # append exactly one EOS
        core = core + [eos_id]
        
        # Pad to max_len with pad_id (0) on the right
        padding_length = max_len - len(core)
        core = core + [pad_id] * padding_length
        
        # attention mask: 1 for real tokens (including EOS), 0 for padding
        attn_core = [1] * (len(ids[:max_len - 1]) + 1) + [0] * padding_length
        
        out_ids.append(core)
        out_mask.append(attn_core)

    return {'input_ids': out_ids, 'attention_mask': out_mask}

def fix_labels(batch):
    batch['labels'] = np.array(batch['DV_n'], dtype=np.float32)
    return batch

In [5]:
# Verify tokenizer adds EOS token
sample = train_df['text'].iloc[0]
tokens = tokenizer(sample, add_special_tokens=True)
decoded = tokenizer.decode(tokens['input_ids'])
print(f'Sample text: {sample[:50]}...')
print(f'Has EOS token: {tokenizer.eos_token_id in tokens["input_ids"]}')
print(f'Token IDs (last 5): {tokens["input_ids"][-5:]}')
print(f'EOS token ID: {tokenizer.eos_token_id}')

Sample text: [C][=C][C][=C][C][=Branch1][Branch1][=C][N][Ring1]...
Has EOS token: False
Token IDs (last 5): [1, 0, 2]
EOS token ID: 3


In [6]:

# Prepare datasets for Trainer
from datasets import Dataset
train_ds = Dataset.from_pandas(train_df)
val_ds = Dataset.from_pandas(val_df)
train_ds = train_ds.map(tokenize_fn, batched=True)
val_ds = val_ds.map(tokenize_fn, batched=True)
train_ds = train_ds.map(fix_labels, batched=True)
val_ds = val_ds.map(fix_labels, batched=True)
train_ds = train_ds.remove_columns(['text','DV_C','DV_n'])
val_ds = val_ds.remove_columns(['text','DV_C','DV_n'])

# No DataCollator needed - sequences are already padded to max_len in tokenize_fn
data_collator = None


Map: 100%|██████████| 200/200 [00:00<00:00, 41429.32 examples/s]


In [7]:
# Unit test: Verify all examples have exactly one EOS token
def validate_eos_tokens(dataset, tokenizer, dataset_name="dataset"):
    """
    Validates that every example in the dataset has exactly one EOS token.
    Raises ValueError if validation fails.
    """
    eos_id = tokenizer.eos_token_id
    if eos_id is None:
        raise ValueError("tokenizer.eos_token_id is None, cannot validate EOS tokens")
    
    eos_counts = []
    sample_size = min(100, len(dataset))  # Check first 100 samples or all if smaller
    
    for i in range(sample_size):
        example = dataset[i]
        if 'input_ids' not in example:
            raise ValueError(f"Example {i} missing 'input_ids'")
        
        input_ids = example['input_ids']
        eos_count = input_ids.count(eos_id)
        eos_counts.append(eos_count)
        
        if eos_count != 1:
            print(f"FAIL: Example {i} has {eos_count} EOS tokens (expected 1)")
            print(f"  Input IDs (last 10): {input_ids[-10:]}")
            print(f"  EOS token ID: {eos_id}")
            raise ValueError(f"{dataset_name}: Example {i} has {eos_count} EOS tokens, expected exactly 1")
    
    # Check uniformity
    unique_counts = set(eos_counts)
    if len(unique_counts) != 1 or unique_counts.pop() != 1:
        raise ValueError(f"{dataset_name}: EOS token counts vary: {unique_counts}")
    
    print(f"✓ {dataset_name} validation PASSED: All {sample_size} samples have exactly 1 EOS token")
    return True

# Run validation on both datasets
print("Validating training dataset...")
validate_eos_tokens(train_ds, tokenizer, "train_ds")

print("\nValidating validation dataset...")
validate_eos_tokens(val_ds, tokenizer, "val_ds")

print("\n✓✓ All datasets passed EOS validation - ready for training!")

Validating training dataset...
✓ train_ds validation PASSED: All 100 samples have exactly 1 EOS token

Validating validation dataset...
✓ val_ds validation PASSED: All 100 samples have exactly 1 EOS token

✓✓ All datasets passed EOS validation - ready for training!


In [ ]:
# Debug: Check what DataCollator does to a batch
print("Testing DataCollator behavior with EOS tokens...")
print(f"EOS token ID: {tokenizer.eos_token_id}")
print(f"PAD token ID: {tokenizer.pad_token_id}")

# Get a small batch
test_batch = [train_ds[i] for i in range(4)]
print(f"\nBefore DataCollator:")
for i, item in enumerate(test_batch):
    ids = item['input_ids']
    eos_count = ids.count(tokenizer.eos_token_id)
    print(f"  Example {i}: len={len(ids)}, EOS count={eos_count}, last token={ids[-1]}")

# Apply collator
# collated = data_collator(test_batch)
# print(f"\nAfter DataCollator:")
# for i in range(len(test_batch)):
#     ids = collated['input_ids'][i].tolist()
#     eos_count = ids.count(tokenizer.eos_token_id)
#     print(f"  Example {i}: len={len(ids)}, EOS count={eos_count}, last 5 tokens={ids[-5:]}")

# print("\n⚠️ If EOS counts changed after DataCollator, that's the problem!")

Testing DataCollator behavior with EOS tokens...
EOS token ID: 3
PAD token ID: 3

Before DataCollator:
  Example 0: len=256, EOS count=1, last token=0
  Example 1: len=256, EOS count=1, last token=0
  Example 2: len=256, EOS count=1, last token=0
  Example 3: len=256, EOS count=1, last token=0


TypeError: 'NoneType' object is not callable

In [9]:

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    if isinstance(preds, tuple):
        preds = preds[0]
    preds = np.squeeze(preds)
    labels = np.squeeze(labels)
    from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
    mse = mean_squared_error(labels, preds)
    return {
        'mse': float(mse),
        'rmse': float(np.sqrt(mse)),
        'mae': float(mean_absolute_error(labels, preds)),
        'r2': float(r2_score(labels, preds)),
    }


In [10]:
# Single GPU training args (GPU 1 only is visible due to CUDA_VISIBLE_DEVICES)
training_args = TrainingArguments(
    output_dir='../data/results_selfies_ted',
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    eval_strategy='epoch',
    save_strategy='no',
    logging_steps=10,
    learning_rate=5e-5,
    remove_unused_columns=False,
    fp16=True,                    # Mixed precision on V100
    gradient_checkpointing=True,  # Memory saving
    dataloader_num_workers=2,
    optim='adamw_torch',
    report_to=[],
    label_names=['labels'],       # Explicitly specify label column
)

# Optional performance tweaks
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.benchmark = True

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print('trainer ready (single GPU mode)')
print(f'Visible GPU count: {torch.cuda.device_count()}')

trainer ready (single GPU mode)
Visible GPU count: 1


In [11]:
# Robust training wrapper with diagnostics
from contextlib import suppress

def print_gpu_mem(prefix=""):
    if torch.cuda.is_available():
        stats = []
        for i in range(torch.cuda.device_count()):
            allocated = torch.cuda.memory_allocated(i) / 1024**2
            reserved = torch.cuda.memory_reserved(i) / 1024**2
            stats.append(f"GPU{i} alloc={allocated:.1f}MB reserved={reserved:.1f}MB")
        print(prefix + " | ".join(stats))

print_gpu_mem("Before training")
try:
    train_results = trainer.train()
    print_gpu_mem("After training")
    print(train_results)
except RuntimeError as e:
    # Common CUDA OOM or AMP errors
    if 'CUDA out of memory' in str(e):
        print('CUDA OOM detected. Suggest lowering batch size further (e.g. 2) or disabling fp16.')
    elif 'cublas' in str(e).lower():
        print('cuBLAS error. Potential driver/library mismatch. Consider reinstalling matching torch + CUDA.')
    else:
        print('RuntimeError during training:', e)
    # Cleanup
    if torch.cuda.is_available():
        with suppress(Exception):
            torch.cuda.empty_cache()
        print_gpu_mem("After cleanup")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 3, 'bos_token_id': None}.


Before trainingGPU0 alloc=1369.9MB reserved=1396.0MB


Epoch,Training Loss,Validation Loss,Mse,Rmse,Mae,R2
1,43.216800,40.885345,40.885345,6.394165,5.398350,-0.000308
2,48.684900,40.891693,40.891693,6.394661,5.401396,-0.000463
3,67.472000,41.134697,41.134697,6.413634,5.333586,-0.006409


After trainingGPU0 alloc=4127.9MB reserved=11900.0MB
TrainOutput(global_step=750, training_loss=51.81923181152344, metrics={'train_runtime': 126.7117, 'train_samples_per_second': 23.676, 'train_steps_per_second': 5.919, 'total_flos': 1630169731584000.0, 'train_loss': 51.81923181152344, 'epoch': 3.0})


In [12]:
# Diagnostic: verify process bound to the intended GPU only
if torch.cuda.is_available():
    current = torch.cuda.current_device()
    print(f'Current CUDA device index (should be 0 now): {current}')
    print(f'Device name: {torch.cuda.get_device_name(current)}')
else:
    print('CUDA not available after restriction.')

Current CUDA device index (should be 0 now): 0
Device name: Quadro GV100


In [ ]:
import IPython
app = IPython.Application.instance()
app.kernel.do_shutdown(True)

{'status': 'ok', 'restart': True}

: 